In [3]:
pip install pgmpy  #Probabilistic Graphical Models


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 38.7 MB/s eta 0:00:00


In [4]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [5]:
model = DiscreteBayesianNetwork([
    ('Age', 'BloodPressure'),
    ('Age', 'Cholesterol'),
    ('BloodPressure', 'HeartDisease'),
    ('Cholesterol', 'HeartDisease'),
    ('ChestPain', 'HeartDisease')
])

In [6]:
cpd_age = TabularCPD(
    variable='Age',
    variable_card=2,
    values=[[0.6],   # Young
            [0.4]]   # Old
)

In [7]:
cpd_bp = TabularCPD(
    variable='BloodPressure',
    variable_card=2,
    values=[[0.7, 0.3],   # Normal
            [0.3, 0.7]],  # High
    evidence=['Age'],
    evidence_card=[2]
)

In [8]:
cpd_chol = TabularCPD(
    variable='Cholesterol',
    variable_card=2,
    values=[[0.8, 0.4],   # Normal
            [0.2, 0.6]],  # High
    evidence=['Age'],
    evidence_card=[2]
)

In [9]:
cpd_cp = TabularCPD(
    variable='ChestPain',
    variable_card=2,
    values=[[0.7],
            [0.3]]
)

In [10]:
cpd_hd = TabularCPD(
    variable='HeartDisease',
    variable_card=2,
    values=[
        [0.95, 0.8, 0.7, 0.4, 0.85, 0.6, 0.5, 0.2],  # No
        [0.05, 0.2, 0.3, 0.6, 0.15, 0.4, 0.5, 0.8]   # Yes
    ],
    evidence=['BloodPressure', 'Cholesterol', 'ChestPain'],
    evidence_card=[2, 2, 2]
)

In [11]:
model.add_cpds(cpd_age, cpd_bp, cpd_chol, cpd_cp, cpd_hd)

In [12]:
model.add_cpds(cpd_age, cpd_bp, cpd_chol, cpd_cp, cpd_hd)

In [13]:
print(model.check_model())

True


In [14]:
inference = VariableElimination(model)

In [15]:
result = inference.query(
    variables=['HeartDisease'],
    evidence={
        'Age': 1,
        'BloodPressure': 1,
        'Cholesterol': 1,
        'ChestPain': 1
    }
)
print(result)

+-----------------+---------------------+
| HeartDisease    |   phi(HeartDisease) |
+=================+=====================+
| HeartDisease(0) |              0.2000 |
+-----------------+---------------------+
| HeartDisease(1) |              0.8000 |
+-----------------+---------------------+


In [18]:
try:
    result = inference.query(
        variables=['HeartDisease'],
        evidence={
            'Age': 0,
            'BloodPressure': 0,
            'Cholesterol': 0,
            'ChestPain': 0
        }
    )
    print(result)
except Exception as e:
    print("Error:", e)

+-----------------+---------------------+
| HeartDisease    |   phi(HeartDisease) |
+=================+=====================+
| HeartDisease(0) |              0.9500 |
+-----------------+---------------------+
| HeartDisease(1) |              0.0500 |
+-----------------+---------------------+


In [16]:
if result.values[1] > result.values[0]:
    print("Heart Disease Detected")
else:
    print("No Heart Disease")

Heart Disease Detected
